# Inference speed: HF Transformers vs vLLM

Benchmark single-prompt latency and throughput on the same
Llama-3.2-1B-Instruct checkpoint across:

1. HF Transformers (eager generation)
2. vLLM with `gpu_memory_utilization=0.2`
3. vLLM with `gpu_memory_utilization=0.7`

Decoding is stochastic (`do_sample=True` / `temperature > 0`) on
both backends, with matching `temperature` and `top_p`. Each timed
iteration uses a fresh seed `base_seed + i` — outputs differ from
run to run but are reproducible across re-executions of the cell.
Token counts vary per iteration since the sampled completions hit
EOS at different points; throughput (tokens / second) is the
robust metric to compare.

Each backend includes one untimed warmup pass to exclude cudagraph
capture / JIT compilation from the latency.

Note: `gpu_memory_utilization` mainly affects vLLM's KV-cache pool
size, which matters for *concurrent* requests. On a single prompt
the two vLLM settings should land within noise of each other.

## Setup

In [1]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)

import gc
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from vllm import LLM, SamplingParams

In [ ]:
# Dataset and model paths
base_dir = '/groups/chichengz/tnn/datasets/'

dataset_dir = base_dir + "/prm800k/math_splits"

# Causal LM under test (swap to Llama-3.2-1B-Instruct to compare)
# llm_dir = base_dir + "/Llama-3.2-1B-Instruct"
llm_dir = base_dir + "Qwen2.5-3B-Instruct"

In [3]:
# Benchmark prompt + decoding config
prompt = (
    r'If $f(x) = \frac{3x-2}{x-2}$, what is the value of '
    r'$f(-2) + f(-1) + f(0)$? Express your answer as a common fraction.'
)
max_new_tokens = 1024
num_runs = 10

# Stochastic decoding — same params on both backends for a fair comparison
temperature = 0.8
top_p = 0.95
base_seed = 123    # iteration i uses seed = base_seed + i

In [4]:
def gpu_mem_used_gb(device=0):
    """Driver-level used GPU memory; sees both PyTorch and vLLM allocs."""
    free, total = torch.cuda.mem_get_info(device)
    return (total - free) / (1024**3)


def measure_inference(
    backend, model, tokenizer, prompt, max_new_tokens, num_runs,
    temperature, top_p, base_seed=123, warmup=1,
):
    """Time `num_runs` stochastic generations and report stats.

    `backend` is "hf" or "vllm". A `warmup` untimed pass absorbs
    cudagraph capture / JIT overhead. Each timed iteration uses
    `base_seed + i` so runs differ but are reproducible across
    re-executions. Only newly generated tokens are counted.
    """
    def _generate(seed):
        if backend == "vllm":
            params = SamplingParams(
                temperature=temperature,
                top_p=top_p,
                max_tokens=max_new_tokens,
                seed=seed,
            )
            out = model.generate(prompt, params, use_tqdm=False)
            tok_ids = out[0].outputs[0].token_ids
            return out[0].outputs[0].text, len(tok_ids)
        else:
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            prompt_len = inputs['input_ids'].shape[1]
            pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
            torch.manual_seed(seed)        # seeds CPU + CUDA RNGs
            with torch.no_grad():
                out_ids = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,
                    temperature=temperature,
                    top_p=top_p,
                    pad_token_id=pad_id,
                )
            new_ids = out_ids[0, prompt_len:]
            return tokenizer.decode(new_ids, skip_special_tokens=True), len(new_ids)

    # Warmup: use a seed outside the timed range so timed runs stay reproducible
    for w in range(warmup):
        _generate(base_seed + 10_000 + w)

    total_time = 0.0
    total_tokens = 0
    text = ""
    for i in range(num_runs):
        start = time.perf_counter()
        text, n_tokens = _generate(base_seed + i)
        total_time += time.perf_counter() - start
        total_tokens += n_tokens

    latency = total_time / num_runs
    throughput = total_tokens / total_time
    avg_tokens = total_tokens / num_runs
    return latency, throughput, avg_tokens, text

## HF Transformers (baseline)

In [5]:
tokenizer = AutoTokenizer.from_pretrained(llm_path)
model_hf = AutoModelForCausalLM.from_pretrained(
    llm_path,
    torch_dtype="float16",
    device_map="cuda:0",
)
model_hf.eval()

print(f'#--- GPU memory used: {gpu_mem_used_gb():.2f} GB')

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

#--- GPU memory used: 8.81 GB


In [6]:
print(model_hf.dtype)
stop

torch.float16


NameError: name 'stop' is not defined

In [ ]:
latency_hf, throughput_hf, avg_tokens_hf, text_hf = measure_inference(
    "hf", model_hf, tokenizer, prompt, max_new_tokens, num_runs,
    temperature=temperature, top_p=top_p, base_seed=base_seed,
)
print(
    f"HF Transformers   - latency: {latency_hf:.4f}s, "
    f"throughput: {throughput_hf:.2f} tok/s, "
    f"avg tokens: {avg_tokens_hf:.1f}"
)

HF Transformers   - latency: 8.6875s, throughput: 59.95 tok/s, avg tokens: 520.8


In [ ]:
# Free HF before loading vLLM so they don't fight over the GPU
del model_hf
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f'#--- GPU memory used: {gpu_mem_used_gb():.2f} GB')

#--- GPU memory used: 6.59 GB


## vLLM with `gpu_memory_utilization=0.2`

Small KV-cache pool — enough headroom for one prompt but not for
concurrent batching at long contexts.

In [ ]:
llm_vllm = LLM(
    model=llm_path,
    tensor_parallel_size=1,
    gpu_memory_utilization=0.2,
    max_model_len=5000,
    dtype="float16",
    seed=123,
)

print(f'#--- GPU memory used: {gpu_mem_used_gb():.2f} GB')

INFO 05-15 12:26:07 [utils.py:233] non-default args: {'dtype': 'float16', 'seed': 123, 'max_model_len': 5000, 'gpu_memory_utilization': 0.2, 'disable_log_stats': True, 'model': '/groups/kjun/tnn/datasets/Llama-3.2-1B-Instruct'}
INFO 05-15 12:26:07 [model.py:549] Resolved architecture: LlamaForCausalLM
WARNING 05-15 12:26:07 [model.py:2016] Casting torch.bfloat16 to torch.float16.
INFO 05-15 12:26:07 [model.py:1678] Using max model len 5000
INFO 05-15 12:26:08 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-15 12:26:08 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 05-15 12:26:11 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


(EngineCore pid=2415171) INFO 05-15 12:26:20 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='/groups/kjun/tnn/datasets/Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='/groups/kjun/tnn/datasets/Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=5000, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_vers

(EngineCore pid=2415171) <frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=2415171) <frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore pid=2415171) INFO 05-15 12:26:24 [weight_utils.py:848] Prefetching checkpoint files into page cache started (in background)
(EngineCore pid=2415171) INFO 05-15 12:26:25 [weight_utils.py:825] Prefetching checkpoint files: 10% (1/1)
(EngineCore pid=2415171) INFO 05-15 12:26:25 [weight_utils.py:843] Prefetching checkpoint files into page cache finished in 0.26s


Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.25s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.25s/it]
(EngineCore pid=2415171) 


(EngineCore pid=2415171) INFO 05-15 12:26:26 [default_loader.py:384] Loading weights took 1.30 seconds
(EngineCore pid=2415171) INFO 05-15 12:26:26 [gpu_model_runner.py:4820] Model loading took 2.32 GiB memory and 2.655830 seconds
(EngineCore pid=2415171) INFO 05-15 12:26:31 [backends.py:1051] Using cache directory: /home/u20/tnguyen9210/.cache/vllm/torch_compile_cache/6ab7c98631/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=2415171) INFO 05-15 12:26:31 [backends.py:1111] Dynamo bytecode transform time: 4.81 s
(EngineCore pid=2415171) INFO 05-15 12:26:35 [backends.py:372] Cache the graph of compile range (1, 8192) for later use
(EngineCore pid=2415171) INFO 05-15 12:26:38 [backends.py:390] Compiling a graph for compile range (1, 8192) takes 6.38 s
(EngineCore pid=2415171) INFO 05-15 12:26:39 [decorators.py:655] saved AOT compiled function to /home/u20/tnguyen9210/.cache/vllm/torch_compile_cache/torch_aot_compile/51a709b45a2511c43101d6e0309e8700808bc13ac87b037d02547d0b18901

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 47.13it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:03<00:00, 11.43it/s]


(EngineCore pid=2415171) INFO 05-15 12:26:55 [gpu_model_runner.py:6046] Graph capturing finished in 5 secs, took 0.28 GiB
(EngineCore pid=2415171) INFO 05-15 12:26:55 [gpu_worker.py:597] CUDA graph pool memory: 0.28 GiB (actual), 0.24 GiB (estimated), difference: 0.04 GiB (13.3%).
(EngineCore pid=2415171) INFO 05-15 12:26:55 [core.py:283] init engine (profile, create kv cache, warmup model) took 29.05 seconds
(EngineCore pid=2415171) INFO 05-15 12:26:57 [vllm.py:790] Asynchronous scheduling is enabled.
#--- GPU memory used: 13.53 GB


In [ ]:
latency_v02, throughput_v02, avg_tokens_v02, text_v02 = measure_inference(
    "vllm", llm_vllm, None, prompt, max_new_tokens, num_runs,
    temperature=temperature, top_p=top_p, base_seed=base_seed,
)
print(
    f"vLLM gpu_mem=0.2  - latency: {latency_v02:.4f}s, "
    f"throughput: {throughput_v02:.2f} tok/s, "
    f"avg tokens: {avg_tokens_v02:.1f}"
)

vLLM gpu_mem=0.2  - latency: 1.9547s, throughput: 250.98 tok/s, avg tokens: 490.6


In [ ]:
# Free the first vLLM engine before reloading at a higher pool size
del llm_vllm
gc.collect()
torch.cuda.empty_cache()
print(f'#--- GPU memory used: {gpu_mem_used_gb():.2f} GB')

(EngineCore pid=2415171) INFO 05-15 12:27:20 [core.py:1210] Shutdown initiated (timeout=0)
(EngineCore pid=2415171) INFO 05-15 12:27:20 [core.py:1233] Shutdown complete


[rank0]:[W515 12:27:20.501761831 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


#--- GPU memory used: 6.59 GB


## vLLM with `gpu_memory_utilization=0.7`

Large KV-cache pool. For a single prompt this should match the
0.2 setting within noise; the difference shows up under concurrent
batching (more sequences live in cache at once).

In [ ]:
llm_vllm = LLM(
    model=llm_path,
    tensor_parallel_size=1,
    gpu_memory_utilization=0.7,
    max_model_len=5000,
    dtype="float16",
    seed=123,
)

print(f'#--- GPU memory used: {gpu_mem_used_gb():.2f} GB')

INFO 05-15 12:27:21 [utils.py:233] non-default args: {'dtype': 'float16', 'seed': 123, 'max_model_len': 5000, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'model': '/groups/kjun/tnn/datasets/Llama-3.2-1B-Instruct'}
INFO 05-15 12:27:21 [model.py:549] Resolved architecture: LlamaForCausalLM
WARNING 05-15 12:27:21 [model.py:2016] Casting torch.bfloat16 to torch.float16.
INFO 05-15 12:27:21 [model.py:1678] Using max model len 5000


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


(EngineCore pid=2415346) INFO 05-15 12:27:30 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='/groups/kjun/tnn/datasets/Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='/groups/kjun/tnn/datasets/Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=5000, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_vers

(EngineCore pid=2415346) <frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=2415346) <frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore pid=2415346) INFO 05-15 12:27:34 [weight_utils.py:825] Prefetching checkpoint files: 10% (1/1)
(EngineCore pid=2415346) INFO 05-15 12:27:34 [weight_utils.py:843] Prefetching checkpoint files into page cache finished in 0.26s


Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.26s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.26s/it]
(EngineCore pid=2415346) 


(EngineCore pid=2415346) INFO 05-15 12:27:35 [default_loader.py:384] Loading weights took 1.30 seconds
(EngineCore pid=2415346) INFO 05-15 12:27:35 [gpu_model_runner.py:4820] Model loading took 2.32 GiB memory and 2.111633 seconds
(EngineCore pid=2415346) INFO 05-15 12:27:38 [backends.py:1051] Using cache directory: /home/u20/tnguyen9210/.cache/vllm/torch_compile_cache/6ab7c98631/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=2415346) INFO 05-15 12:27:38 [backends.py:1111] Dynamo bytecode transform time: 2.57 s
(EngineCore pid=2415346) INFO 05-15 12:27:39 [backends.py:285] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 0.590 s
(EngineCore pid=2415346) INFO 05-15 12:27:39 [decorators.py:305] Directly load AOT compilation from path /home/u20/tnguyen9210/.cache/vllm/torch_compile_cache/torch_aot_compile/51a709b45a2511c43101d6e0309e8700808bc13ac87b037d02547d0b18901944/rank_0_0/model
(EngineCore pid=2415346) INFO 05-15 12:27:39 [monitor.py:4

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 47.67it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:00<00:00, 40.34it/s]


(EngineCore pid=2415346) INFO 05-15 12:27:43 [gpu_model_runner.py:6046] Graph capturing finished in 2 secs, took 0.28 GiB
(EngineCore pid=2415346) INFO 05-15 12:27:43 [gpu_worker.py:597] CUDA graph pool memory: 0.28 GiB (actual), 0.24 GiB (estimated), difference: 0.04 GiB (13.3%).
(EngineCore pid=2415346) INFO 05-15 12:27:43 [core.py:283] init engine (profile, create kv cache, warmup model) took 7.77 seconds
(EngineCore pid=2415346) INFO 05-15 12:27:44 [vllm.py:790] Asynchronous scheduling is enabled.
#--- GPU memory used: 29.44 GB


In [ ]:
latency_v07, throughput_v07, avg_tokens_v07, text_v07 = measure_inference(
    "vllm", llm_vllm, None, prompt, max_new_tokens, num_runs,
    temperature=temperature, top_p=top_p, base_seed=base_seed,
)
print(
    f"vLLM gpu_mem=0.7  - latency: {latency_v07:.4f}s, "
    f"throughput: {throughput_v07:.2f} tok/s, "
    f"avg tokens: {avg_tokens_v07:.1f}"
)

vLLM gpu_mem=0.7  - latency: 1.9540s, throughput: 251.07 tok/s, avg tokens: 490.6


## Summary

In [ ]:
header = f"{'Backend':<22}{'Latency (s)':>14}{'Tok/s':>12}{'Avg tok':>12}"
print(header)
print('-' * len(header))
rows = [
    ('HF Transformers',  latency_hf,  throughput_hf,  avg_tokens_hf),
    ('vLLM gpu_mem=0.2', latency_v02, throughput_v02, avg_tokens_v02),
    ('vLLM gpu_mem=0.7', latency_v07, throughput_v07, avg_tokens_v07),
]
for name, lat, tput, ntok in rows:
    print(f"{name:<22}{lat:>14.4f}{tput:>12.2f}{ntok:>12.1f}")

print(f"\nSample completion (HF, last run):\n{text_hf[:400]}...")

Backend                  Latency (s)       Tok/s     Avg tok
------------------------------------------------------------
HF Transformers               8.6875       59.95       520.8
vLLM gpu_mem=0.2              1.9547      250.98       490.6
vLLM gpu_mem=0.7              1.9540      251.07       490.6

Sample completion (HF, last run):
 To evaluate $f(-2)$, plug in $x=-2$ into $f(x)$. Similarly, to evaluate $f(-1)$, plug in $x=-1$ into $f(x)$. To evaluate $f(0)$, plug in $x=0$ into $f(x)$. Evaluate each of these expressions. \begin{align*} f(-2) &amp;= \frac{3(-2)-2}{(-2)-2}\\ &= \frac{-8-2}{-4}\\ &= \frac{-10}{-4}\\ &= \frac{5}{2} \end{align*} \begin{align*} f(-1) &amp;= \frac{3(-1)-2}{(-1)-2}\\ &= \frac{-3-2}{-3}\\ &= \frac{-5...
